# CTGAN & TVAE — Génération de Données Synthétiques
## Pipeline complet corrigé v2

Ce notebook implémente CTGAN et TVAE pour générer des données synthétiques
à partir de la base **Home Credit Default Risk**.

### Corrections v2
- **Fix 1** : Fonction DP corrigée — sensibilité normalisée P1-P99 (plus d'outliers)
- **Fix 2** : Calibration du taux de défaut CTGAN après génération
- **Fix 3** : QI numériques ajoutés pour cohérence avec le notebook bayésien
- **Fix 4** : TSTR sur données DP ajouté
- **Fix 5** : DCR ajoutée pour mesurer le risque de réidentification CTGAN/TVAE
- **Fix 6** : Classifieur TSTR unifié (LogisticRegression) pour comparaison cohérente


## 0. Installation des packages


In [ ]:
# À exécuter une seule fois dans Colab (3-5 minutes)
!pip install ctgan sdv diffprivlib -q
!pip install scikit-learn xgboost -q


## 1. Imports & Chargement des données


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression  # unifié avec notebook bayésien
from sklearn.ensemble import RandomForestClassifier

print('Imports OK')


In [ ]:
# Chargement de la base originale
app_train = pd.read_csv('./application_train.csv')
print(f'Shape original : {app_train.shape}')
print(f'Taux de défaut : {app_train["TARGET"].mean():.2%}')


## 2. Préparation des données


In [ ]:
# Sélection des variables
COLS_NUMERIQUES = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

COLS_CATEGORIELLES = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START'
]

CIBLE = 'TARGET'
TOUTES_COLS = COLS_NUMERIQUES + COLS_CATEGORIELLES + [CIBLE]

df = app_train[TOUTES_COLS].copy()
print(f'Shape après sélection : {df.shape}')


In [ ]:
# Nettoyage
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

df['AGE_YEARS']      = -df['DAYS_BIRTH']    / 365
df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365
df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)

COLS_NUMERIQUES = [c for c in COLS_NUMERIQUES if c not in ['DAYS_BIRTH', 'DAYS_EMPLOYED']]
COLS_NUMERIQUES += ['AGE_YEARS', 'YEARS_EMPLOYED']

print(f'Shape après nettoyage : {df.shape}')


In [ ]:
# Imputation des valeurs manquantes
imputer_num = SimpleImputer(strategy='median')
df[COLS_NUMERIQUES] = imputer_num.fit_transform(df[COLS_NUMERIQUES])

imputer_cat = SimpleImputer(strategy='most_frequent')
df[COLS_CATEGORIELLES] = imputer_cat.fit_transform(df[COLS_CATEGORIELLES])

print(f'Valeurs manquantes restantes : {df.isnull().sum().sum()}')


In [ ]:
# Échantillon de développement
SAMPLE_SIZE  = 50_000
RANDOM_STATE = 42

if len(df) <= SAMPLE_SIZE:
    df_sample = df.copy().reset_index(drop=True)
else:
    df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Taille du jeu utilisé : {df_sample.shape}')
print(f'Taux de défaut        : {df_sample[CIBLE].mean():.2%}')


## 3. CTGAN — Conditional Tabular GAN

**Principe** : CTGAN est un GAN adapté aux données tabulaires. Il utilise un *conditional vector*
pour gérer les variables catégorielles et le déséquilibre des classes. Le générateur et le
discriminateur s'entraînent en adversarial jusqu'à ce que le générateur produise des données
indiscernables des vraies.


In [ ]:
from ctgan import CTGAN

print('=' * 60)
print('ENTRAÎNEMENT DU CTGAN (300 epochs)')
print('=' * 60)

ctgan_model = CTGAN(
    epochs=300,
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    verbose=True
)

print('\nEntraînement en cours...')
ctgan_model.fit(
    df_sample,
    discrete_columns=COLS_CATEGORIELLES + [CIBLE]
)
print('\nCTGAN entraîné !')


In [ ]:
# Génération
print('=' * 60)
print('GÉNÉRATION DES DONNÉES SYNTHÉTIQUES CTGAN')
print('=' * 60)

df_ctgan_raw = ctgan_model.sample(len(df_sample))

taux_reel  = df_sample[CIBLE].mean()
taux_ctgan_raw = df_ctgan_raw[CIBLE].mean()
print(f'Taux de défaut réel        : {taux_reel:.4f} ({taux_reel*100:.2f}%)')
print(f'Taux de défaut CTGAN brut  : {taux_ctgan_raw:.4f} ({taux_ctgan_raw*100:.2f}%)')
print(f'Écart                      : {abs(taux_ctgan_raw-taux_reel)*100:.2f}%')
print(f'Shape : {df_ctgan_raw.shape}')


In [ ]:
# FIX 2 — Calibration du taux de défaut CTGAN
# CTGAN peut générer un taux de défaut légèrement différent du réel.
# On rééchantillonne pour forcer exactement le bon taux.

def calibrate_target_rate(df_synth, target_col, target_rate, random_state=42):
    """
    Rééchantillonne df_synth pour que target_col ait exactement target_rate.
    Retourne un DataFrame de même taille avec le bon taux.
    """
    n_total   = len(df_synth)
    n_defaut  = int(round(n_total * target_rate))
    n_sain    = n_total - n_defaut

    df_def = df_synth[df_synth[target_col] == 1]
    df_sai = df_synth[df_synth[target_col] == 0]

    # Si pas assez de défauts, on sur-échantillonne avec replacement
    df_def_s = df_def.sample(n=n_defaut, replace=(len(df_def) < n_defaut),
                              random_state=random_state)
    df_sai_s = df_sai.sample(n=n_sain,   replace=(len(df_sai) < n_sain),
                              random_state=random_state)

    df_cal = pd.concat([df_def_s, df_sai_s]).sample(frac=1, random_state=random_state)
    return df_cal.reset_index(drop=True)


df_ctgan = calibrate_target_rate(df_ctgan_raw, CIBLE, taux_reel, RANDOM_STATE)

print(f'Taux de défaut après calibration : {df_ctgan[CIBLE].mean():.4f}')
print(f'Shape après calibration          : {df_ctgan.shape}')
print()
print('Aperçu :')
print(df_ctgan.head())


In [ ]:
# Évaluation statistique CTGAN
print('=' * 60)
print('ÉVALUATION STATISTIQUE — CTGAN')
print('=' * 60)

# Test Chi-2 sur TARGET
contingency_table = pd.crosstab(
    pd.Series(['Réel']*len(df_sample) + ['Synthétique']*len(df_ctgan)),
    pd.Series(list(df_sample[CIBLE]) + list(df_ctgan[CIBLE]))
)
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f'\nTest Chi-2 (distribution TARGET) :')
print(f'  Chi-2 : {chi2:.4f}  |  P-value : {p_value:.6f}')
if p_value > 0.05:
    print(f'  → Distributions IDENTIQUES (p > 0.05) ✓')
else:
    print(f'  → Distributions DIFFÉRENTES (p < 0.05)')

# KL divergence variables catégorielles
print(f'\nKL Divergence par colonne catégorielle :')
kl_divs_cat = {}
for col in COLS_CATEGORIELLES + [CIBLE]:
    real_dist  = df_sample[col].value_counts(normalize=True)
    synth_dist = df_ctgan[col].value_counts(normalize=True)
    all_cats   = real_dist.index.union(synth_dist.index)
    real_dist  = real_dist.reindex(all_cats, fill_value=1e-10)
    synth_dist = synth_dist.reindex(all_cats, fill_value=1e-10)
    kl = stats.entropy(real_dist, synth_dist)
    kl_divs_cat[col] = kl
    qual = '✓ EXCELLENT' if kl < 0.01 else ('✓ BON' if kl < 0.05 else
           ('⚠ ACCEPTABLE' if kl < 0.1 else '✗ MAUVAIS'))
    print(f'  {col:30s} KL={kl:.4f}  {qual}')


In [ ]:
# Visualisation CTGAN
cols_viz = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AGE_YEARS',
            'EXT_SOURCE_2', 'YEARS_EMPLOYED', 'CNT_CHILDREN']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), cols_viz):
    sns.kdeplot(df_sample[col], ax=ax, label='Réel',  color='steelblue', fill=True, alpha=0.4)
    sns.kdeplot(df_ctgan[col],  ax=ax, label='CTGAN', color='tomato',    fill=True, alpha=0.4)
    ax.set_title(col)
    ax.legend()
plt.suptitle('CTGAN — Comparaison des distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('ctgan_distributions.png', dpi=100, bbox_inches='tight')
print('Graphique sauvegardé : ctgan_distributions.png')
plt.show()


## 4. TVAE — Tabular Variational Autoencoder

**Principe** : TVAE encode les données dans un espace latent continu (distribution gaussienne),
puis les décode pour générer de nouvelles observations. Plus stable que CTGAN à entraîner.


In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

# Métadonnées SDV
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_sample)

# Correction des types catégoriels
for col in COLS_CATEGORIELLES + [CIBLE]:
    metadata.update_column(col, sdtype='categorical')

print('Metadata détectées et corrigées')


In [ ]:
print('=' * 60)
print('ENTRAÎNEMENT DU TVAE (300 epochs, batch=128, emb=256)')
print('=' * 60)

tvae_model = TVAESynthesizer(
    metadata,
    epochs=300,
    batch_size=128,           # Petit batch : meilleure capture des défauts (~8%)
    compress_dims=(256, 256), # Grand embedding : plus d'information conservée
    decompress_dims=(256, 256),
    embedding_dim=256,
    verbose=True
)

print('\nEntraînement en cours...')
tvae_model.fit(df_sample)
print('\nTVAE entraîné !')


In [ ]:
# Génération TVAE
df_tvae_raw = tvae_model.sample(num_rows=len(df_sample))

taux_tvae_raw = df_tvae_raw[CIBLE].mean()
print(f'Taux de défaut réel      : {df_sample[CIBLE].mean():.4f}')
print(f'Taux de défaut TVAE brut : {taux_tvae_raw:.4f}')
print(f'Écart                    : {abs(taux_tvae_raw - df_sample[CIBLE].mean())*100:.2f}%')
print(f'Shape : {df_tvae_raw.shape}')

# Calibration du taux de défaut TVAE (même logique que CTGAN)
df_tvae = calibrate_target_rate(df_tvae_raw, CIBLE, df_sample[CIBLE].mean(), RANDOM_STATE)
print(f'\nTaux après calibration : {df_tvae[CIBLE].mean():.4f} ✓')


In [ ]:
# Évaluation statistique TVAE
print('=' * 60)
print('ÉVALUATION STATISTIQUE — TVAE')
print('=' * 60)

print('\nKL Divergence par colonne catégorielle :')
kl_divs_tvae = {}
for col in COLS_CATEGORIELLES + [CIBLE]:
    real_dist  = df_sample[col].value_counts(normalize=True)
    synth_dist = df_tvae[col].value_counts(normalize=True)
    all_cats   = real_dist.index.union(synth_dist.index)
    real_dist  = real_dist.reindex(all_cats, fill_value=1e-10)
    synth_dist = synth_dist.reindex(all_cats, fill_value=1e-10)
    kl = stats.entropy(real_dist, synth_dist)
    kl_divs_tvae[col] = kl
    qual = '✓ EXCELLENT' if kl < 0.01 else ('✓ BON' if kl < 0.05 else
           ('⚠ ACCEPTABLE' if kl < 0.1 else '✗ MAUVAIS'))
    print(f'  {col:30s} KL={kl:.4f}  {qual}')


In [ ]:
# Visualisation TVAE vs Réel
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), cols_viz):
    sns.kdeplot(df_sample[col], ax=ax, label='Réel', color='steelblue', fill=True, alpha=0.4)
    sns.kdeplot(df_tvae[col],   ax=ax, label='TVAE', color='green',     fill=True, alpha=0.4)
    ax.set_title(col)
    ax.legend()
plt.suptitle('TVAE — Comparaison des distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('tvae_distributions.png', dpi=100, bbox_inches='tight')
print('Graphique sauvegardé : tvae_distributions.png')
plt.show()


## 5. Confidentialité : k-anonymat, l-diversity, Differential Privacy, DCR

### 5a. k-anonymat

**Définition** : Un dataset satisfait le k-anonymat si chaque combinaison de
quasi-identifiants apparaît dans au moins k lignes.


In [ ]:
# k-ANONYMAT
# FIX 3 — Deux sets de QI :
# QI_CAT  : catégoriels (approche originale)
# QI_NUM  : numériques discrétisés (cohérent avec notebook bayésien)

def compute_k_anonymity(df, quasi_identifiers):
    groups = df.groupby(quasi_identifiers).size()
    return groups.min(), groups


def compute_k_anonymity_numeric(df, num_qi_cols, n_bins=10):
    """Discrétise les QI numériques avant le calcul du k-anonymat."""
    df_gen = df.copy()
    for col in num_qi_cols:
        if df[col].nunique() > n_bins:
            df_gen[col] = pd.cut(df[col], bins=n_bins, precision=0).astype(str)
        else:
            df_gen[col] = df[col].astype(str)
    return compute_k_anonymity(df_gen, num_qi_cols)


# QI catégoriels (approche originale)
QI_CAT = ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE']

# QI numériques (cohérent avec notebook bayésien)
QI_NUM = ['AGE_YEARS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS']

# ── QI catégoriels ──────────────────────────────────────────────────────────
k_reel_cat,  grp_reel_cat  = compute_k_anonymity(df_sample, QI_CAT)
k_ctgan_cat, grp_ctgan_cat = compute_k_anonymity(df_ctgan,  QI_CAT)
k_tvae_cat,  grp_tvae_cat  = compute_k_anonymity(df_tvae,   QI_CAT)

# ── QI numériques ───────────────────────────────────────────────────────────
k_reel_num,  grp_reel_num  = compute_k_anonymity_numeric(df_sample, QI_NUM)
k_ctgan_num, grp_ctgan_num = compute_k_anonymity_numeric(df_ctgan,  QI_NUM)
k_tvae_num,  grp_tvae_num  = compute_k_anonymity_numeric(df_tvae,   QI_NUM)

print('=== k-ANONYMAT — QI Catégoriels ===')
print(f'  Réel  : k_min = {k_reel_cat}')
print(f'  CTGAN : k_min = {k_ctgan_cat}')
print(f'  TVAE  : k_min = {k_tvae_cat}')
print()
print('=== k-ANONYMAT — QI Numériques (AGE / CNT_CHILDREN / CNT_FAM) ===')
print(f'  Réel  : k_min = {k_reel_num}')
print(f'  CTGAN : k_min = {k_ctgan_num}')
print(f'  TVAE  : k_min = {k_tvae_num}')

# Graphique distribution des groupes (QI catégoriels)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (grp, label, col) in zip(axes, [
    (grp_reel_cat,  f'Réel — k_min={k_reel_cat}',   'steelblue'),
    (grp_ctgan_cat, f'CTGAN — k_min={k_ctgan_cat}',  'tomato'),
    (grp_tvae_cat,  f'TVAE — k_min={k_tvae_cat}',    'green'),
]):
    grp.hist(bins=30, ax=ax, color=col, edgecolor='white')
    ax.set_title(label)
    ax.set_xlabel('Taille du groupe')
plt.suptitle('Distribution des groupes k-anonymat (QI catégoriels)', fontsize=13)
plt.tight_layout()
plt.show()


### 5b. l-diversity

**Définition** : Extension du k-anonymat. Un groupe est l-diverse si l'attribut sensible
(TARGET = défaut) prend au moins l valeurs distinctes dans chaque groupe.


In [ ]:
# l-DIVERSITY
def compute_l_diversity(df, quasi_identifiers, sensitive_col):
    l_values = df.groupby(quasi_identifiers)[sensitive_col].nunique()
    return l_values.min(), l_values


l_reel_cat,  l_dist_reel_cat  = compute_l_diversity(df_sample, QI_CAT, CIBLE)
l_ctgan_cat, l_dist_ctgan_cat = compute_l_diversity(df_ctgan,  QI_CAT, CIBLE)
l_tvae_cat,  l_dist_tvae_cat  = compute_l_diversity(df_tvae,   QI_CAT, CIBLE)

print('=== l-DIVERSITY (QI catégoriels) ===')
print(f'  Réel  : l_min = {l_reel_cat}')
print(f'  CTGAN : l_min = {l_ctgan_cat}')
print(f'  TVAE  : l_min = {l_tvae_cat}')

for label, l_dist in [
    ('Réel',  l_dist_reel_cat),
    ('CTGAN', l_dist_ctgan_cat),
    ('TVAE',  l_dist_tvae_cat),
]:
    n_risque = (l_dist == 1).sum()
    pct = n_risque / len(l_dist) * 100
    print(f'  {label:5s} : {n_risque} groupes homogènes sur {len(l_dist)} ({pct:.1f}%) — risque d\'inférence')


### 5c. Differential Privacy (ε-DP)

**Principe** : La DP garantit mathématiquement qu'aucun individu ne peut être identifié
à partir des données publiées. Le bruit est calibré par le paramètre ε.

**Correction v2** : sensibilité normalisée par P1-P99 (résistante aux outliers).


In [ ]:
from diffprivlib import tools as dp_tools

# Impact de ε sur les statistiques (démonstration pédagogique)
col_dp   = 'AMT_INCOME_TOTAL'
true_mean = df_sample[col_dp].mean()

# FIX 1 — Bornes robustes P1-P99 pour éviter l'explosion due aux outliers
bound_lo = float(np.percentile(df_sample[col_dp], 1))
bound_hi = float(np.percentile(df_sample[col_dp], 99))
bounds   = (bound_lo, bound_hi)

print(f'Statistiques réelles pour {col_dp} :')
print(f'  Moyenne vraie : {true_mean:,.0f}')
print(f'  Bornes P1-P99 : [{bound_lo:,.0f} — {bound_hi:,.0f}]')
print()
print(f'Impact de ε sur la précision (bornes robustes) :')
print(f'{"ε":<8} {"Moyenne DP":>15} {"Erreur relative":>18}')
print('-' * 45)

for eps in [0.1, 0.5, 1.0, 5.0, 10.0]:
    mean_dp = dp_tools.mean(df_sample[col_dp].values, epsilon=eps, bounds=bounds)
    erreur  = abs(mean_dp - true_mean) / true_mean
    print(f'ε={eps:<5} {mean_dp:>15,.0f} {erreur:>17.2%}')


In [ ]:
# FIX 1 — Fonction DP corrigée : normalisation P1-P99
# L'ancienne version utilisait min/max bruts → bruit explosif sur les outliers
# (ex. AMT_INCOME_TOTAL max = 117M€ → bruit de 117M€ à ε=1)

def apply_dp_noise(df_synthetic, numeric_cols, epsilon=1.0, ref_df=None):
    """
    Differential Privacy par mécanisme de Laplace avec normalisation P1-P99.
    - Normalise chaque variable dans [0,1] selon les percentiles 1%-99% du réel
    - Applique bruit Laplace(0, 1/epsilon) dans l'espace normalisé (sensibilité=1)
    - Reconvertit à l'échelle originale
    - Préserve TARGET (variable binaire non bruitée)
    """
    rng   = np.random.default_rng(42)
    df_dp = df_synthetic.copy()
    ref   = ref_df if ref_df is not None else df_synthetic

    for col in numeric_cols:
        if col == CIBLE:
            continue

        # Bornes robustes P1-P99 (insensibles aux outliers extrêmes)
        vmin = float(np.percentile(ref[col], 1))
        vmax = float(np.percentile(ref[col], 99))
        rng_ = vmax - vmin
        if rng_ == 0:
            continue

        # Normalisation dans [0,1] + clip avant bruit
        x_norm  = np.clip((df_dp[col].values - vmin) / rng_, 0.0, 1.0)

        # Bruit Laplace — sensibilité = 1 (espace normalisé)
        noise   = rng.laplace(loc=0, scale=1.0 / epsilon, size=len(df_dp))
        x_noisy = np.clip(x_norm + noise, 0.0, 1.0)

        # Retour à l'échelle originale
        df_dp[col] = x_noisy * rng_ + vmin

    # TARGET conservée intacte
    if CIBLE in df_dp.columns:
        df_dp[CIBLE] = df_synthetic[CIBLE].values
    return df_dp


COLS_NUM_SYNTH = [c for c in COLS_NUMERIQUES if c in df_ctgan.columns]

# Application sur CTGAN et TVAE avec ε=1.0
df_ctgan_dp = apply_dp_noise(df_ctgan, COLS_NUM_SYNTH, epsilon=1.0, ref_df=df_sample)
df_tvae_dp  = apply_dp_noise(df_tvae,  COLS_NUM_SYNTH, epsilon=1.0, ref_df=df_sample)

print('Vérification post-DP (moyennes AMT_INCOME_TOTAL) :')
print(f'  Réel              : {df_sample["AMT_INCOME_TOTAL"].mean():>12,.0f} €')
print(f'  CTGAN (sans DP)   : {df_ctgan["AMT_INCOME_TOTAL"].mean():>12,.0f} €')
print(f'  CTGAN + DP (ε=1)  : {df_ctgan_dp["AMT_INCOME_TOTAL"].mean():>12,.0f} €')
print(f'  TVAE (sans DP)    : {df_tvae["AMT_INCOME_TOTAL"].mean():>12,.0f} €')
print(f'  TVAE + DP (ε=1)   : {df_tvae_dp["AMT_INCOME_TOTAL"].mean():>12,.0f} €')
print()
print('Taux de défaut après DP :')
print(f'  CTGAN + DP : {df_ctgan_dp[CIBLE].mean():.4f}')
print(f'  TVAE  + DP : {df_tvae_dp[CIBLE].mean():.4f}')


### 5d. Distance to Closest Record (DCR) — risque de réidentification

**Principe** : Pour chaque individu réel, on mesure la distance euclidienne
à l'individu synthétique le plus proche. Une DCR faible = risque de quasi-clone.

- **DCR médiane** : protection typique — plus c'est haut, mieux c'est
- **% quasi-clones** : individus réels avec un synthétique à distance < 0.01 (risque direct)


In [ ]:
# FIX 5 — DCR ajoutée (absente de la version précédente)
def compute_dcr(df_real, df_synth, feature_cols, n_sample=2000, random_state=42):
    """
    Distance to Closest Record : pour chaque individu réel,
    distance euclidienne à l'individu synthétique le plus proche.
    Toutes les variables sont normalisées P1-P99 pour être comparables.
    """
    rng = np.random.default_rng(random_state)
    n   = min(n_sample, len(df_real), len(df_synth))

    # Sélectionner uniquement les colonnes numériques communes
    num_cols = [c for c in feature_cols
                if c in df_real.columns and c in df_synth.columns
                and pd.api.types.is_numeric_dtype(df_real[c])]

    idx_r = rng.choice(len(df_real),  n, replace=False)
    idx_s = rng.choice(len(df_synth), n, replace=False)

    X_r = df_real.iloc[idx_r][num_cols].values.astype(float)
    X_s = df_synth.iloc[idx_s][num_cols].values.astype(float)

    # Normalisation P1-P99 basée sur le réel
    vmin = np.percentile(X_r, 1, axis=0)
    vmax = np.percentile(X_r, 99, axis=0)
    rng_ = np.where(vmax - vmin > 0, vmax - vmin, 1)
    X_r_n = np.clip((X_r - vmin) / rng_, 0, 1)
    X_s_n = np.clip((X_s - vmin) / rng_, 0, 1)

    distances = []
    for i in range(len(X_r_n)):
        diffs = X_s_n - X_r_n[i]
        dists = np.sqrt((diffs ** 2).sum(axis=1))
        distances.append(dists.min())

    d = np.array(distances)
    return {
        'dcr_median': float(np.median(d)),
        'dcr_p5':     float(np.percentile(d, 5)),
        'pct_near0':  float((d < 0.01).mean() * 100),
    }


all_cols = COLS_NUMERIQUES + COLS_CATEGORIELLES

# Référence synth→synth
half = len(df_ctgan) // 2
dcr_ss = compute_dcr(df_ctgan.iloc[:half], df_ctgan.iloc[half:], all_cols)

print('=' * 65)
print('RISQUE DE RÉIDENTIFICATION — DCR (Distance to Closest Record)')
print('(DCR médiane élevée = meilleure protection)')
print('=' * 65)
print(f'\nRéférence synth→synth (CTGAN) : DCR médiane={dcr_ss["dcr_median"]:.4f}')
print()

dcr_results = {}
for label, df_s in [
    ('CTGAN',        df_ctgan),
    ('TVAE',         df_tvae),
    ('CTGAN + DP',   df_ctgan_dp),
    ('TVAE + DP',    df_tvae_dp),
]:
    dcr = compute_dcr(df_sample, df_s, all_cols)
    dcr_results[label] = dcr
    print(f'{label:12s} : DCR médiane={dcr["dcr_median"]:.4f} '
          f'| P5={dcr["dcr_p5"]:.4f} '
          f'| % quasi-clones={dcr["pct_near0"]:.2f}%')

print()
print('Interprétation : DCR > référence synth→synth = protection adéquate.')


In [ ]:
# Résumé confidentialité
print('\n=== RÉSUMÉ CONFIDENTIALITÉ ===')
print(f'{"Métrique":<28} {"Réel":>10} {"CTGAN":>10} {"TVAE":>10}')
print('-' * 62)
print(f'{"k-anonymat catégoriel (k_min)":<28} {k_reel_cat:>10} {k_ctgan_cat:>10} {k_tvae_cat:>10}')
print(f'{"k-anonymat numérique (k_min)":<28} {k_reel_num:>10} {k_ctgan_num:>10} {k_tvae_num:>10}')
print(f'{"l-diversity (l_min)":<28} {l_reel_cat:>10} {l_ctgan_cat:>10} {l_tvae_cat:>10}')
print(f'{"DCR médiane":<28} {"—":>10} {dcr_results["CTGAN"]["dcr_median"]:>10.4f} {dcr_results["TVAE"]["dcr_median"]:>10.4f}')
print(f'{"DCR médiane + DP (ε=1)":<28} {"—":>10} {dcr_results["CTGAN + DP"]["dcr_median"]:>10.4f} {dcr_results["TVAE + DP"]["dcr_median"]:>10.4f}')


## 6. Évaluation comparative TSTR / TRTS

**TSTR (Train on Synthetic, Test on Real)** : si AUC ≈ TRTR (baseline), les données
synthétiques preservent le signal prédictif.

**FIX 6** : Classifieur unifié — LogisticRegression (cohérent avec notebook bayésien).


In [ ]:
# Préparation pour l'évaluation
def prepare_for_model(df, cat_cols, num_cols, target):
    """Encode les catégorielles et retourne X, y."""
    df_enc = df.copy()
    for col in cat_cols:
        if col in df_enc.columns:
            le = LabelEncoder()
            df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    feature_cols = [c for c in num_cols + cat_cols if c in df_enc.columns]
    X = df_enc[feature_cols].values
    y = df_enc[target].values.astype(int)
    return X, y, feature_cols


COLS_COMMUNES_NUM = [c for c in COLS_NUMERIQUES    if c in df_ctgan.columns and c in df_tvae.columns]
COLS_COMMUNES_CAT = [c for c in COLS_CATEGORIELLES if c in df_ctgan.columns and c in df_tvae.columns]

# Split du réel (20% test)
df_reel_train, df_reel_test = train_test_split(
    df_sample, test_size=0.2, random_state=RANDOM_STATE, stratify=df_sample[CIBLE]
)

X_real_train, y_real_train, feat_cols = prepare_for_model(
    df_reel_train, COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)
X_real_test,  y_real_test,  _         = prepare_for_model(
    df_reel_test,  COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)
X_ctgan,      y_ctgan,      _         = prepare_for_model(
    df_ctgan,      COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)
X_tvae,       y_tvae,       _         = prepare_for_model(
    df_tvae,       COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)
X_ctgan_dp,   y_ctgan_dp,   _         = prepare_for_model(
    df_ctgan_dp,   COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)
X_tvae_dp,    y_tvae_dp,    _         = prepare_for_model(
    df_tvae_dp,    COLS_COMMUNES_CAT, COLS_COMMUNES_NUM, CIBLE)

print(f'Features utilisées : {len(feat_cols)} ({feat_cols[:3]}...)')


In [ ]:
# FIX 6 — Classifieur TSTR unifié : LogisticRegression (cohérent avec notebook bayésien)
print('Calcul des AUC TSTR...')
print()

clf_params = dict(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)

def tstr_auc(X_train, y_train, X_test, y_test):
    clf = LogisticRegression(**clf_params)
    clf.fit(X_train, y_train)
    return roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])


# TRTR baseline
auc_trtr          = tstr_auc(X_real_train, y_real_train, X_real_test, y_real_test)

# TSTR
auc_tstr_ctgan    = tstr_auc(X_ctgan,    y_ctgan,    X_real_test, y_real_test)
auc_tstr_tvae     = tstr_auc(X_tvae,     y_tvae,     X_real_test, y_real_test)
auc_tstr_ctgan_dp = tstr_auc(X_ctgan_dp, y_ctgan_dp, X_real_test, y_real_test)
auc_tstr_tvae_dp  = tstr_auc(X_tvae_dp,  y_tvae_dp,  X_real_test, y_real_test)

# TRTS
clf_real = LogisticRegression(**clf_params)
clf_real.fit(X_real_train, y_real_train)
auc_trts_ctgan = roc_auc_score(y_ctgan, clf_real.predict_proba(X_ctgan)[:, 1])
auc_trts_tvae  = roc_auc_score(y_tvae,  clf_real.predict_proba(X_tvae)[:, 1])

print(f'TRTR baseline     : AUC={auc_trtr:.4f}  | Gini={2*auc_trtr-1:.4f}')
print()
print(f'TSTR CTGAN        : AUC={auc_tstr_ctgan:.4f}  | Gini={2*auc_tstr_ctgan-1:.4f}  '
      f'(Δ={auc_tstr_ctgan-auc_trtr:+.4f})')
print(f'TSTR TVAE         : AUC={auc_tstr_tvae:.4f}  | Gini={2*auc_tstr_tvae-1:.4f}  '
      f'(Δ={auc_tstr_tvae-auc_trtr:+.4f})')
print(f'TSTR CTGAN + DP   : AUC={auc_tstr_ctgan_dp:.4f}  | Gini={2*auc_tstr_ctgan_dp-1:.4f}  '
      f'(Δ={auc_tstr_ctgan_dp-auc_trtr:+.4f})')
print(f'TSTR TVAE  + DP   : AUC={auc_tstr_tvae_dp:.4f}  | Gini={2*auc_tstr_tvae_dp-1:.4f}  '
      f'(Δ={auc_tstr_tvae_dp-auc_trtr:+.4f})')
print()
print(f'TRTS CTGAN        : AUC={auc_trts_ctgan:.4f}')
print(f'TRTS TVAE         : AUC={auc_trts_tvae:.4f}')


In [ ]:
# Visualisation comparative finale
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── G1 : AUC TSTR comparatif ─────────────────────────────────────────────────
ax = axes[0]
labels  = ['TRTR\n(Baseline)', 'CTGAN\n(TSTR)', 'TVAE\n(TSTR)',
           'CTGAN+DP\n(TSTR)', 'TVAE+DP\n(TSTR)']
aucs    = [auc_trtr, auc_tstr_ctgan, auc_tstr_tvae,
           auc_tstr_ctgan_dp, auc_tstr_tvae_dp]
colors  = ['steelblue', 'tomato', 'green', '#FF8C00', '#228B22']
bars = ax.bar(labels, aucs, color=colors, width=0.5, edgecolor='white')
ax.axhline(y=auc_trtr, color='navy', linestyle='--', alpha=0.7, label='Baseline réel')
ax.set_ylim(max(0.4, min(aucs)-0.05), min(1.0, max(aucs)+0.08))
ax.set_ylabel('AUC')
ax.set_title('AUC TSTR — Comparaison complète')
ax.legend(fontsize=8)
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── G2 : DCR médiane ─────────────────────────────────────────────────────────
ax = axes[1]
dcr_labels = ['CTGAN', 'TVAE', 'CTGAN+DP', 'TVAE+DP']
dcr_vals   = [dcr_results['CTGAN']['dcr_median'],    dcr_results['TVAE']['dcr_median'],
              dcr_results['CTGAN + DP']['dcr_median'], dcr_results['TVAE + DP']['dcr_median']]
dcr_colors = ['tomato', 'green', '#FF8C00', '#228B22']
bars2 = ax.bar(dcr_labels, dcr_vals, color=dcr_colors, width=0.5, edgecolor='white')
ax.axhline(dcr_ss['dcr_median'], color='gray', linestyle='--', linewidth=1.5,
           label=f'Réf. synth→synth ({dcr_ss["dcr_median"]:.4f})')
ax.set_title('DCR médiane (↑ = meilleure protection)')
ax.set_ylabel('Distance to Closest Record')
ax.legend(fontsize=8)
for bar, val in zip(bars2, dcr_vals):
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

# ── G3 : Scatter AUC vs DCR ──────────────────────────────────────────────────
ax = axes[2]
scatter_data = [
    ('CTGAN',      auc_tstr_ctgan,    dcr_results['CTGAN']['dcr_median'],     'tomato'),
    ('TVAE',       auc_tstr_tvae,     dcr_results['TVAE']['dcr_median'],      'green'),
    ('CTGAN+DP',   auc_tstr_ctgan_dp, dcr_results['CTGAN + DP']['dcr_median'],'#FF8C00'),
    ('TVAE+DP',    auc_tstr_tvae_dp,  dcr_results['TVAE + DP']['dcr_median'], '#228B22'),
]
for name, auc_v, dcr_v, col in scatter_data:
    ax.scatter(dcr_v, auc_v, color=col, s=150, zorder=5)
    ax.annotate(name, (dcr_v, auc_v), textcoords='offset points',
                xytext=(8, 4), fontsize=9)
ax.axhline(auc_trtr, color='navy', linestyle='--', alpha=0.6, label='Baseline TRTR')
ax.axvline(dcr_ss['dcr_median'], color='gray', linestyle='--', alpha=0.6,
           label='Réf. synth→synth')
ax.set_xlabel('DCR médiane (↑ = meilleure confidentialité)')
ax.set_ylabel('AUC TSTR (↑ = meilleure utilité)')
ax.set_title('Compromis utilité / confidentialité')
ax.legend(fontsize=8)

plt.suptitle('Comparaison CTGAN vs TVAE — Utilité & Confidentialité', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('tstr_comparison.png', dpi=100, bbox_inches='tight')
print('Graphique sauvegardé : tstr_comparison.png')
plt.show()


In [ ]:
# Courbes ROC comparatives CTGAN vs TVAE vs Réel
fig, ax = plt.subplots(figsize=(8, 7))

roc_data = [
    (X_real_train, y_real_train, X_real_test,  y_real_test, 'TRTR (Baseline)', 'steelblue', '-'),
    (X_ctgan,      y_ctgan,      X_real_test,  y_real_test, 'TSTR CTGAN',      'tomato',    '--'),
    (X_tvae,       y_tvae,       X_real_test,  y_real_test, 'TSTR TVAE',       'green',     '-.'),
]

for X_tr, y_tr, X_te, y_te, label, color, ls in roc_data:
    clf = LogisticRegression(**clf_params)
    clf.fit(X_tr, y_tr)
    fpr, tpr, _ = roc_curve(y_te, clf.predict_proba(X_te)[:, 1])
    auc_v = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2,
            label=f'{label} (AUC={auc_v:.4f} | Gini={2*auc_v-1:.4f})')

ax.plot([0,1],[0,1], 'k--', linewidth=0.8, label='Aléatoire (AUC=0.5)')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbes ROC — TRTR / TSTR CTGAN / TSTR TVAE')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=100, bbox_inches='tight')
print('Graphique sauvegardé : roc_curves.png')
plt.show()


In [ ]:
# Tableau récapitulatif final
print('\n' + '=' * 75)
print('TABLEAU COMPARATIF FINAL — CTGAN vs TVAE vs Bayésien')
print('=' * 75)
print(f'\n{"Métrique":<35} {"Bayésien":>10} {"CTGAN":>10} {"TVAE":>10}')
print('-' * 70)
print(f'{"TRTR (baseline réel)":<35} {auc_trtr:>10.4f} {auc_trtr:>10.4f} {auc_trtr:>10.4f}')
print(f'{"AUC TSTR":<35} {"~0.692":>10} {auc_tstr_ctgan:>10.4f} {auc_tstr_tvae:>10.4f}')
print(f'{"Gini TSTR":<35} {"~0.383":>10} {2*auc_tstr_ctgan-1:>10.4f} {2*auc_tstr_tvae-1:>10.4f}')
print(f'{"TRTS":<35} {"~0.949":>10} {auc_trts_ctgan:>10.4f} {auc_trts_tvae:>10.4f}')
print(f'{"Taux de défaut généré":<35} {"8.05%":>10} {df_ctgan[CIBLE].mean():>10.2%} {df_tvae[CIBLE].mean():>10.2%}')
print(f'{"DCR médiane (brut)":<35} {"0.217":>10} {dcr_results["CTGAN"]["dcr_median"]:>10.4f} {dcr_results["TVAE"]["dcr_median"]:>10.4f}')
print(f'{"DCR médiane (+ DP ε=1)":<35} {"0.682":>10} {dcr_results["CTGAN + DP"]["dcr_median"]:>10.4f} {dcr_results["TVAE + DP"]["dcr_median"]:>10.4f}')
print(f'{"k-anonymat (numérique)":<35} {"k=5":>10} {k_ctgan_num:>10} {k_tvae_num:>10}')
print()
print('Note : valeurs Bayésien issues du notebook bayésien_v5_confidentialite')


In [ ]:
# Sauvegarde des données synthétiques
# Décommenter pour sauvegarder :
# df_ctgan.to_csv('synthetic_ctgan.csv', index=False)
# df_tvae.to_csv('synthetic_tvae.csv', index=False)
# df_ctgan_dp.to_csv('synthetic_ctgan_dp.csv', index=False)
# df_tvae_dp.to_csv('synthetic_tvae_dp.csv', index=False)

print('Données synthétiques disponibles :')
print(f'  df_ctgan       : {df_ctgan.shape}    — taux défaut {df_ctgan[CIBLE].mean():.2%}')
print(f'  df_tvae        : {df_tvae.shape}    — taux défaut {df_tvae[CIBLE].mean():.2%}')
print(f'  df_ctgan_dp    : {df_ctgan_dp.shape}    — CTGAN + DP (ε=1.0)')
print(f'  df_tvae_dp     : {df_tvae_dp.shape}    — TVAE  + DP (ε=1.0)')
print()
print('Pour sauvegarder, décommenter les lignes ci-dessus.')
